In [1]:
# # Appendix D.2: two-parameter counterexample
#
# C_N(1) > C_H(1) = 3/2
#
# C_MI(2) = C_N(2) = C_H(2) = 3/4
#
# %%
import numpy as np

np.set_printoptions(precision=12, suppress=True)


In [2]:
# ## 1. The one-copy model


# %%
S = np.diag([.5,.5,0.,0.])

A = np.array([
    [0,0],
    [1,0]
], dtype=float)

B = (4/5) * np.eye(2)
C = np.eye(2)

Z = np.block([
    [A, B],
    [C, np.zeros((2,2))]
])

Sx = (10/23) * (Z @ S + S @ Z.T)
Sy = (-10j/7) * (Z @ S - S @ Z.T)

X = (Z + Z.T)/2
Y = (Z - Z.T)/(2j)

assert np.allclose(Sx, Sx.conj().T)
assert np.allclose(Sy, Sy.conj().T)

assert np.isclose(np.trace(S @ Z), 0)

assert np.allclose(
    [
        [np.trace(d @ op) for op in [X,Y]]
        for d in [Sx,Sy]
    ],
    np.eye(2)
)

print("One-copy state:")
print(S)

One-copy state:
[[0.5 0.  0.  0. ]
 [0.  0.5 0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]]


In [3]:
# ## 2. The one-copy Holevo bound
#
# For two parameters,
#
# F_H(Z) = max(q_+(Z), q_-(Z))
#
# where
#
# q_+(Z) = Tr(S Z^dagger Z)
# q_-(Z) = Tr(S Z Z^dagger).
#
# The manuscript proves that the constructed Z
# minimises q_+ over locally unbiased estimators.
#
# In particular, the cross term vanishes for
# every locally unbiased perturbation.
#
# Hence C_H = 3/2.


# %%
qplus = np.trace(
    S @ Z.conj().T @ Z
).real

qminus = np.trace(
    S @ Z @ Z.conj().T
).real

assert np.isclose(qplus, 1.5)
assert qminus < qplus

print("q_plus =", qplus)
print("q_minus =", qminus)
print("Holevo bound = 3/2")


q_plus = 1.5
q_minus = 1.1400000000000001
Holevo bound = 3/2


In [4]:
# ## 3. Why the one-copy Nagaoka bound is larger
#
# Every Holevo optimiser has form
#
# Z' = [[A, B+E],
#       [C, F]]
#
# with Tr(E)=0.
#
# NH equality would require
#
# D - (B+E)(B+E)^dagger >= 0
#
# where D=diag(2,0).
#
# The (2,2) entry forces E_22=-4/5.
# Tr(E)=0 then forces E_11=4/5.
#
# Hence (B+E)_11=8/5 and
#
# (D-(B+E)(B+E)^dagger)_11
# <= 2-(8/5)^2 = -14/25.
#
# The necessary positivity condition fails.


# %%
D = A.T @ A + C.T @ C - A @ A.T

assert np.allclose(D, np.diag([2.,0.]))

necessary_upper_bound = 2 - (8/5)**2

assert np.isclose(
    necessary_upper_bound,
    -14/25
)

print("Necessary PSD entry <=", necessary_upper_bound)
print("Thus C_N(1) > C_H(1) = 3/2")


Necessary PSD entry <= -0.5600000000000005
Thus C_N(1) > C_H(1) = 3/2


In [5]:
# ## 4. Construct the two-copy normal estimator
#
# Define Z_P = Z_+ direct_sum Z_-
#
# in the symmetric and antisymmetric sectors.
#
# The parameter x=-sqrt(2177)/50 makes
#
# Z_P^dagger Z_P = Z_P Z_P^dagger.


# %%
a = 1/np.sqrt(2)
b = 3*np.sqrt(2)/5
r = np.sqrt(7)/5
x = -np.sqrt(2177)/50

Ap = np.array([
    [0,0,0],
    [a,0,0],
    [0,a,0]
])

Bp = np.array([
    [b,0,0,0,r,0,0],
    [0,a,0,0,0,0,0],
    [0,0,0,0,0,0,0]
])

Cp = np.array([
    [a,0,0],
    [0,a,0],
    [0,0,0],
    [0,0,a],
    [0,0,0],
    [0,0,0],
    [0,0,0]
])

F5 = np.array([
    [x,-a/2,b,0,-x/np.sqrt(14)],
    [a/2,15*x/14,0,a,-1/(4*np.sqrt(7))],
    [a,0,0,-a,0],
    [0,a,0,0,0],
    [x/np.sqrt(14),-1/(4*np.sqrt(7)),
     -np.sqrt(7)/5,0,-11*x/14]
])

Fp = np.zeros((7,7))
Fp[:5,:5] = F5

Zplus = np.block([
    [Ap,Bp],
    [Cp,Fp]
])

Zminus = np.zeros((6,6))
Zminus[:2,:2] = [[0,a],[a,0]]

assert Zplus.shape == (10,10)
assert Zminus.shape == (6,6)



In [6]:
# ## 5. Transform to the standard two-copy basis
#
# Construct the symmetric and antisymmetric basis
# vectors as specified in Appendix D.2.


# %%
def ket(i,j):
    v = np.zeros(16)
    v[4*i+j] = 1
    return v


def pair(i,j,sign=1):
    return (
        ket(i,j) + sign*ket(j,i)
    ) / np.sqrt(2)


plus_basis = [
    ket(0,0),
    pair(0,1),
    ket(1,1),
    pair(2,0),
    (pair(2,1)+pair(3,0))/np.sqrt(2),
    (pair(2,1)-pair(3,0))/np.sqrt(2),
    pair(3,1),
    ket(2,2),
    pair(2,3),
    ket(3,3)
]

minus_basis = [
    pair(0,1,-1),
    (pair(2,1,-1)-pair(3,0,-1))/np.sqrt(2),
    pair(2,0,-1),
    (pair(2,1,-1)+pair(3,0,-1))/np.sqrt(2),
    pair(3,1,-1),
    pair(2,3,-1)
]

U = np.column_stack(
    plus_basis + minus_basis
)

assert np.allclose(U.T @ U, np.eye(16))

Zblock = np.zeros((16,16), dtype=complex)
Zblock[:10,:10] = Zplus
Zblock[10:,10:] = Zminus

ZP = U @ Zblock @ U.T

normality = np.linalg.norm(
    ZP.conj().T @ ZP - ZP @ ZP.conj().T
)

assert normality < 1e-12

print("Normality residual =", normality)

Normality residual = 1.2230976154919677e-15


In [7]:
# ## 6. Check local unbiasedness and cost


# %%
S2 = np.kron(S,S)

Sx2 = (
    np.kron(Sx,S)
    + np.kron(S,Sx)
)

Sy2 = (
    np.kron(Sy,S)
    + np.kron(S,Sy)
)

trace0 = np.trace(S2 @ ZP)
trace_x = np.trace(Sx2 @ ZP)
trace_y = np.trace(Sy2 @ ZP)

cost = np.trace(
    S2 @ ZP.conj().T @ ZP
).real

assert abs(trace0) < 1e-12
assert abs(trace_x-1) < 1e-12
assert abs(trace_y-1j) < 1e-12
assert abs(cost-3/4) < 1e-12

print("Tr(S2 ZP) =", trace0)
print("Tr(Sx2 ZP) =", trace_x)
print("Tr(Sy2 ZP) =", trace_y)
print("Two-copy cost =", cost)


Tr(S2 ZP) = 0j
Tr(Sx2 ZP) = (0.9999999999999998+0j)
Tr(Sy2 ZP) = 0.9999999999999999j
Two-copy cost = 0.7499999999999996


In [8]:
# ## 7. Construct the attaining projective measurement
#
# Since Z_P is normal, its Hermitian real and
# imaginary parts commute.
#
# A common eigenbasis gives a joint projective
# measurement.
#
# The eigenvalues lambda give the estimates
# (Re(lambda), Im(lambda)).


# %%
Xp = (ZP + ZP.conj().T)/2
Yp = (ZP - ZP.conj().T)/(2j)

assert np.linalg.norm(
    Xp @ Yp - Yp @ Xp
) < 1e-12

# Generic Hermitian combination of commuting operators.
_, P = np.linalg.eigh(
    Xp + 0.387123*Yp
)

lam = np.diag(
    P.conj().T @ ZP @ P
)

assert np.linalg.norm(
    P.conj().T @ ZP @ P - np.diag(lam)
) < 1e-10

p = np.array([
    np.vdot(P[:,k], S2 @ P[:,k]).real
    for k in range(16)
])

px = np.array([
    np.vdot(P[:,k], Sx2 @ P[:,k]).real
    for k in range(16)
])

py = np.array([
    np.vdot(P[:,k], Sy2 @ P[:,k]).real
    for k in range(16)
])

assert abs(p.sum()-1) < 1e-12
assert abs(np.dot(p,lam)) < 1e-12
assert abs(np.dot(px,lam)-1) < 1e-12
assert abs(np.dot(py,lam)-1j) < 1e-12

measurement_cost = np.dot(
    p, np.abs(lam)**2
)

assert abs(measurement_cost-3/4) < 1e-12

print("PVM probability sum =", p.sum())
print("PVM mean-square cost =", measurement_cost)
print("C_MI(2) = C_N(2) = C_H(2) = 3/4")

PVM probability sum = 1.0000000000000007
PVM mean-square cost = 0.7500000000000004
C_MI(2) = C_N(2) = C_H(2) = 3/4
